In [ ]:
# Install pandas and numpy if not already installed
%pip install pandas numpy

import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [ ]:
df = pd.read_csv('adult.csv', index_col=0)

# Remove rows with '?' in any string column
str_cols = df.select_dtypes('string').columns
df = df[~(df[str_cols] == '?').any(axis=1)]

# Data correctness check: salary column vs salary K$
# '>50K' should have salary K$ > 50, '<=50K' should have salary K$ <= 50
inconsistent = (
    ((df['salary'] == '>50K') & (df['salary K$'] <= 50)) |
    ((df['salary'] == '<=50K') & (df['salary K$'] > 50))
)
print(f"Rows removed due to '?': {32561 - len(df)}")
print(f"Inconsistent salary rows: {inconsistent.sum()}")
print(f"Clean dataset shape: {df.shape}")
df.head()

# Task 1
Print the count of men and women in the dataset.

In [1]:
sex_counts = df['sex'].value_counts()
print(sex_counts.to_string())

# Task 2
Find the average age of men in dataset

In [ ]:
avg_age_men = df.loc[df['sex'] == 'Male', 'age'].mean()
print(f"Average age of men: {avg_age_men:.2f}")

# Task 3
Get the percentage of people from Poland (native-country)

In [ ]:
poland_pct = (df['native-country'] == 'Poland').mean() * 100
print(f"Percentage of people from Poland: {poland_pct:.2f}%")

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [2]:
age_stats = df.groupby('salary')['age'].agg(['mean', 'std'])
print(age_stats.to_string())

# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [3]:
higher_education = {'Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate'}

no_higher_high_salary = (
    ~df['education'].isin(higher_education) & (df['salary'] == '>50K')
)
count = no_higher_high_salary.sum()
print(f"People without higher education but earning >50K: {count}")
print(f"Such people exist: {count > 0}")

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [ ]:
age_by_education = df.groupby('education')['age'].describe()
print(age_by_education.to_string())

# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [5]:
men = df[df['sex'] == 'Male'].copy()
men['married'] = men['marital-status'].str.startswith('Married')

salary_comparison = men.groupby('married')['salary'].value_counts(normalize=True).unstack() * 100
salary_comparison.index = salary_comparison.index.map({True: 'Married', False: 'Not Married'})
salary_comparison.index.name = 'Marital Status'
print(salary_comparison.round(2).to_string())
print()

married_50k = salary_comparison.loc['Married', '>50K']
not_married_50k = salary_comparison.loc['Not Married', '>50K']
print(f"Married men earning >50K:     {married_50k:.2f}%")

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [6]:
max_hours = df['hours-per-week'].max()
count_max_hours = (df['hours-per-week'] == max_hours).sum()
print(f"Max hours per week: {max_hours}")
print(f"Number of people working {max_hours} hours/week: {count_max_hours}")

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [7]:
# Encode categorical columns numerically for correlation analysis
df_encoded = df.copy()
df_encoded['salary_num'] = (df['salary'] == '>50K').astype(int)
df_encoded['sex_num'] = (df['sex'] == 'Male').astype(int)
df_encoded['married_num'] = df['marital-status'].str.startswith('Married').astype(int)

numeric_df = df_encoded[['age', 'hours-per-week', 'salary K$', 'salary_num', 'sex_num', 'married_num']]
corr = numeric_df.corr().round(3)
print("Correlation matrix:")
print(corr.to_string())
print()

# Highlight strong correlations (|r| > 0.3)
print("Strong correlations (|r| > 0.3):")
for col1 in corr.columns:
    for col2 in corr.columns:
        if col1 < col2:
            r = corr.loc[col1, col2]
            if abs(r) > 0.3:
                direction = 'positive' if r > 0 else 'negative'
                print(f"  {col1} <-> {col2}: r={r:.3f} ({direction})")